# 🧠 NeuroScan AI — U-Net GPU Training (Google Colab)
### Run every cell top-to-bottom. All fixes are pre-applied.

| Step | What |
|------|------|
| 1 | GPU check |
| 2 | Mount Google Drive |
| 3 | Upload project code zip |
| 4 | Install packages + auto-patch dataset.py |
| 5 | Upload MRI dataset |
| 6 | Generate pseudo masks |
| 7 | Train U-Net (~20 min on T4) |
| 8 | Save to Drive + download .pt file |

**Before running:** `Runtime → Change runtime type → T4 GPU → Save`

## Step 1 — Verify GPU

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Go to Runtime -> Change runtime type -> T4 GPU")
print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
print(f"CUDA   : {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")
print("OK")

## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/NeuroScan_UNet'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Drive mounted. Outputs -> {DRIVE_DIR}")

## Step 3 — Upload Project Code

The zip `BrainTumorAI_code.zip` was already created on your PC at:
`C:\Users\HP\Desktop\PRO(B)\BrainTumorAI\BrainTumorAI_code.zip`

Run the cell below → file picker opens → select that zip.

In [ ]:
from google.colab import files
import zipfile, os, shutil, sys

PROJECT = '/content/BrainTumorAI'
os.makedirs(PROJECT, exist_ok=True)

print("Select BrainTumorAI_code.zip when the picker opens...")
uploaded = files.upload()

for fname in uploaded:
    print(f"Extracting: {fname}")
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('/tmp/extracted/')
    os.remove(fname)

extracted = '/tmp/extracted'
items    = os.listdir(extracted)
subdirs  = [d for d in items if os.path.isdir(os.path.join(extracted, d))]
topfiles = [f for f in items if os.path.isfile(os.path.join(extracted, f))]
src = os.path.join(extracted, subdirs[0]) if (len(subdirs)==1 and not topfiles) else extracted

def _copytree(s, d):
    os.makedirs(d, exist_ok=True)
    for item in os.listdir(s):
        ss = os.path.join(s, item)
        dd = os.path.join(d, item)
        if os.path.isdir(ss):
            _copytree(ss, dd)
        else:
            shutil.copy2(ss, dd)

_copytree(src, PROJECT)
shutil.rmtree('/tmp/extracted', ignore_errors=True)

os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

required = ['app.py','inference','visualization','segmentation','training','report']
missing  = [r for r in required if not os.path.exists(f'{PROJECT}/{r}')]
if missing:
    print(f"MISSING: {missing}  <- re-upload the zip")
else:
    print("All files present!")
    os.system('ls /content/BrainTumorAI/')

## Step 4 — Install Packages & Auto-Patch Code

This cell:
1. Installs timm, albumentations, opencv, reportlab (no gradio — not needed)
2. Patches `dataset.py` to scan class subfolders (glioma/, meningioma/, etc.)

In [ ]:
import subprocess, sys, os

# Install packages
pkgs = [
    'timm==0.9.12',
    'albumentations==1.3.1',
    'opencv-python-headless',
    'reportlab',
]
for pkg in pkgs:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                       capture_output=True, text=True)
    print(f"{'OK  ' if r.returncode==0 else 'FAIL'} {pkg}")

# Patch dataset.py: iterdir() -> rglob() so it finds images in subfolders
ds_path = '/content/BrainTumorAI/segmentation/dataset.py'
with open(ds_path, 'r') as f:
    src = f.read()

OLD = 'p for p in self.image_dir.iterdir()'
NEW = "p for p in self.image_dir.rglob('*')"

if OLD in src:
    with open(ds_path, 'w') as f:
        f.write(src.replace(OLD, NEW))
    print("Patched dataset.py: rglob now scans subfolders")
elif NEW in src:
    print("dataset.py already patched")
else:
    print("WARNING: pattern not found in dataset.py — check manually")

# Quick import test
import importlib
for mod in ['timm', 'albumentations', 'cv2']:
    try:
        importlib.import_module(mod)
        print(f"  import {mod} OK")
    except Exception as e:
        print(f"  import {mod} FAILED: {e}")

print("\nStep 4 complete — continue to Step 5")

## Step 5 — Upload Dataset

**Option A** (cell below): Upload `USE-Me Test.zip` from your PC
**Option B** (second cell): Copy from Google Drive if already there

Dataset structure must be:
```
USE-Me Test/
  glioma/
  meningioma/
  notumor/
  pituitary/
```

In [ ]:
# OPTION A: Upload from PC
# Right-click "USE-Me Test" folder on PC -> Send to -> Compressed
from google.colab import files
import zipfile, os

print("Select 'USE-Me Test.zip' when picker opens...")
uploaded = files.upload()
for fname in uploaded:
    print(f"Extracting: {fname}")
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('/content/BrainTumorAI/')
    os.remove(fname)

# Optionally upload checkpoints too
print("\nSelect 'checkpoints.zip' if you have one (Cancel to skip)...")
try:
    uploaded2 = files.upload()
    for fname in uploaded2:
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('/content/BrainTumorAI/')
        os.remove(fname)
except Exception:
    print("Checkpoints upload skipped")

os.system('ls /content/BrainTumorAI/')

In [ ]:
# OPTION B: Copy from Google Drive
# Use if dataset is at: MyDrive/NeuroScan_UNet/USE-Me Test/
import shutil, os

DRIVE_DATA = '/content/drive/MyDrive/NeuroScan_UNet'
for name, dst in [
    ('USE-Me Test', '/content/BrainTumorAI/USE-Me Test'),
    ('checkpoints',  '/content/BrainTumorAI/checkpoints'),
]:
    src = f'{DRIVE_DATA}/{name}'
    if not os.path.exists(src):
        print(f"Not in Drive: {src}")
    elif os.path.exists(dst):
        print(f"Already exists: {dst}")
    else:
        shutil.copytree(src, dst)
        print(f"Copied: {dst}")

In [ ]:
# Verify dataset
from pathlib import Path
root  = Path('/content/BrainTumorAI/USE-Me Test')
total = 0
for cls in ['glioma', 'meningioma', 'notumor', 'pituitary']:
    p = root / cls
    n = len(list(p.glob('*.*'))) if p.exists() else 0
    flag = 'OK' if n > 0 else 'MISSING'
    print(f"  {cls:12s}: {n:4d} images  {flag}")
    total += n
print(f"\n  Total: {total} images")
assert total > 0, "No images found! Run Option A or B above first."

## Step 6 — Generate Pseudo Masks via Grad-CAM

Converts Grad-CAM heatmaps to binary masks for U-Net training.
If no checkpoint found → runs in demo mode (seeded masks — training still works).

In [ ]:
import sys, os, logging, warnings
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger('colab')

os.chdir('/content/BrainTumorAI')
sys.path.insert(0, '/content/BrainTumorAI')

import timm

DEVICE      = torch.device('cuda')
NUM_CLASSES = 4
IMAGE_SIZE  = 224
ENS_WEIGHTS = {"efficientnet": 0.4, "resnet_cbam": 0.3, "densenet": 0.3}
SEARCH_PATHS = [
    '/content/BrainTumorAI/checkpoints',
    '/content/drive/MyDrive/BrainTumorAI/checkpoints',
    '/content/drive/MyDrive/NeuroScan_UNet/checkpoints',
]
_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# Model definitions (no app.py import — avoids gradio/huggingface crash)
class ChannelAttention(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        m = max(c // r, 8)
        self.fc = nn.Sequential(
            nn.Linear(c, m, bias=False), nn.ReLU(True), nn.Linear(m, c, bias=False))
    def forward(self, x):
        B, C, H, W = x.shape
        a = torch.sigmoid(self.fc(x.mean([2,3])) + self.fc(x.amax([2,3])))
        return x * a.view(B, C, 1, 1)

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, 7, padding=3, bias=False)
    def forward(self, x):
        return x * torch.sigmoid(self.conv(torch.cat([x.mean(1,True), x.amax(1,True)], 1)))

class CBAM(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.ca = ChannelAttention(c, r); self.sa = SpatialAttention()
    def forward(self, x): return self.sa(self.ca(x))

class ResNetCBAM(nn.Module):
    def __init__(self, nc, pretrained=False):
        super().__init__()
        b = timm.create_model('resnet50', pretrained=pretrained, num_classes=0, global_pool='')
        self.conv1=b.conv1; self.bn1=b.bn1; self.act1=b.act1; self.maxpool=b.maxpool
        self.layer1=b.layer1; self.layer2=b.layer2; self.layer3=b.layer3; self.layer4=b.layer4
        self.cbam3=CBAM(1024); self.cbam4=CBAM(2048)
        self.pool=nn.AdaptiveAvgPool2d(1)
        self.head=nn.Sequential(
            nn.Dropout(0.4), nn.Linear(2048,512), nn.ReLU(True),
            nn.Dropout(0.2), nn.Linear(512, nc))
    def forward(self, x):
        x=self.act1(self.bn1(self.conv1(x))); x=self.maxpool(x)
        x=self.layer1(x); x=self.layer2(x)
        x=self.cbam3(self.layer3(x)); x=self.cbam4(self.layer4(x))
        return self.head(self.pool(x).flatten(1))

class Ensemble(nn.Module):
    def __init__(self, models, weights):
        super().__init__()
        self.models=nn.ModuleDict(models); self.weights=weights
        self.temps=nn.ParameterDict({k:nn.Parameter(torch.ones(1)) for k in models})
    def forward(self, x):
        t = None
        for k, m in self.models.items():
            p = F.softmax(m(x) / self.temps[k].clamp(min=0.1), dim=1)
            t = self.weights.get(k, 1.) * p if t is None else t + self.weights.get(k, 1.) * p
        return t
    def individual_probs(self, x):
        return {k: F.softmax(m(x)/self.temps[k].clamp(min=0.1),dim=1).cpu().numpy()
                for k, m in self.models.items()}

class MockEnsemble:
    def predict(self, tensor):
        a=tensor.cpu().numpy(); seed=int((a.mean()*1e4+a.std()*1e3)%1e6)
        rng=np.random.RandomState(seed%100000); dom=rng.randint(0,NUM_CLASSES)
        b=rng.dirichlet(np.ones(NUM_CLASSES)*0.3); b[dom]+=rng.uniform(.4,.65)
        probs=b/b.sum()
        return probs, {n:probs for n in ['efficientnet','resnet_cbam','densenet']}

def _best_ckpt(folder):
    ckpts = list(Path(folder).glob('best_*.pt'))
    if not ckpts: return None
    def _auc(p):
        try: return float(p.stem.split('auc')[-1])
        except: return 0.
    return max(ckpts, key=_auc)

def _remap_keys(sd, mname):
    out = {}
    for k, v in sd.items():
        nk = k[len('backbone.'):] if k.startswith('backbone.') else k
        if mname in ('efficientnet','densenet') and nk.startswith('head.'): continue
        if 'cbam' in nk and 'ca.fc' in nk and v.dim()==4: v=v.squeeze(-1).squeeze(-1)
        out[nk] = v
    return out

def load_model():
    fns = {
        'efficientnet': lambda nc: timm.create_model('efficientnet_b3.ra2_in1k', pretrained=False, num_classes=nc),
        'resnet_cbam':  lambda nc: ResNetCBAM(nc),
        'densenet':     lambda nc: timm.create_model('densenet121', pretrained=False, num_classes=nc),
    }
    for base_str in SEARCH_PATHS:
        base = Path(base_str)
        if not base.exists(): continue
        loaded = {}
        for mname, mfn in fns.items():
            ckpt = _best_ckpt(base / mname)
            if ckpt is None: continue
            try:
                m = mfn(NUM_CLASSES)
                raw = torch.load(ckpt, map_location=DEVICE, weights_only=False)['model_state_dict']
                try:   m.load_state_dict(raw, strict=True)
                except: m.load_state_dict(_remap_keys(raw, mname), strict=False)
                m.eval(); loaded[mname] = m.to(DEVICE)
            except Exception as e:
                logger.warning(f'{mname}: {e}')
        if loaded:
            ens = Ensemble(loaded, {k:ENS_WEIGHTS[k] for k in loaded}).to(DEVICE)
            ens.eval()
            return ens, 'live' if len(loaded)==3 else 'partial'
    return MockEnsemble(), 'demo'

def preprocess(image):
    try:
        if isinstance(image, np.ndarray):
            img = image.copy().astype(np.uint8)
            if img.ndim==2: img=cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
            elif img.shape[2]==4: img=cv2.cvtColor(img, cv2.COLOR_RGBA2RGB)
        elif isinstance(image, Image.Image):
            img = np.array(image.convert('RGB'), dtype=np.uint8)
        else: return None
        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE)).astype(np.float32) / 255.
        img = (img - _MEAN) / _STD
        return torch.from_numpy(img.transpose(2,0,1)).unsqueeze(0).float()
    except: return None

print("Loading classification model...")
model_obj, model_mode = load_model()
print(f"Model mode: {model_mode}")

In [ ]:
from inference.gradcam import EnsembleGradCAM

gradcam_engine = EnsembleGradCAM(model_obj, model_mode)
print("GradCAM engine ready")

_DUMMY = {
    'class':'glioma','class_label':'Glioma','confidence':0.85,
    'probabilities':{'glioma':0.85,'meningioma':0.05,'notumor':0.05,'pituitary':0.05},
    'individual':{}
}

def gradcam_fn(img_np: np.ndarray) -> np.ndarray:
    t = preprocess(img_np)
    if t is None:
        return np.zeros((14,14), dtype=np.float32)
    result = gradcam_engine.generate(t.to(DEVICE), _DUMMY, img_np)
    return result['cam']

In [ ]:
from segmentation.pseudo_masks import build_pseudo_dataset
import shutil, os

PSEUDO_MASK_DIR = '/content/BrainTumorAI/data/pseudo_masks'
os.makedirs(PSEUDO_MASK_DIR, exist_ok=True)

print("Generating pseudo masks...")
n = build_pseudo_dataset(
    images_dir   = '/content/BrainTumorAI/USE-Me Test',
    output_dir   = PSEUDO_MASK_DIR,
    gradcam_fn   = gradcam_fn,
    threshold    = 0.40,
    skip_notumor = True,
    save_npy     = True,
    save_png     = True,
    verbose      = True,
)
print(f"Generated {n} pseudo masks")

drive_masks = f'{DRIVE_DIR}/pseudo_masks'
if os.path.exists(drive_masks):
    shutil.rmtree(drive_masks)
shutil.copytree(PSEUDO_MASK_DIR, drive_masks)
print(f"Backed up to Drive: {drive_masks}")

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

mask_files = sorted(Path(PSEUDO_MASK_DIR).glob('*.png'))[:8]
if not mask_files:
    print("No mask PNGs found — check dataset upload")
else:
    fig, axes = plt.subplots(2, 4, figsize=(16,8), facecolor='#0f172a')
    for ax, mf in zip(axes.flat, mask_files):
        m = cv2.imread(str(mf), cv2.IMREAD_GRAYSCALE)
        ax.imshow(m, cmap='gray')
        ax.set_title(mf.stem[:20], color='#00d4ff', fontsize=7)
        ax.axis('off')
    plt.suptitle('Pseudo Masks (white=tumour)', color='white')
    plt.tight_layout()
    plt.savefig(f'{DRIVE_DIR}/pseudo_mask_preview.png', dpi=80,
                bbox_inches='tight', facecolor='#0f172a')
    plt.show()
    print("Preview saved to Drive")

## Step 7 — Train U-Net on GPU (~20 min on T4)

| Setting | Value | Note |
|---------|-------|------|
| base_filters | 64 | 31M params |
| batch_size | 16 | fits T4 VRAM |
| epochs | 80 | ~20 min |
| image_size | 256 | high res |

In [ ]:
import os
CONFIG = {
    'images_dir'  : '/content/BrainTumorAI/USE-Me Test',
    'pseudo_cache': '/content/BrainTumorAI/data/pseudo_masks',
    'image_size'  : 256,
    'in_channels' : 3,  # RGB — dataset returns 3-channel tensors
    'val_split'   : 0.15,
    'base_filters': 64,
    'bilinear'    : True,
    'epochs'      : 80,
    'batch_size'  : 16,
    'lr'          : 1e-4,
    'bce_alpha'   : 0.5,
    'threshold'   : 0.5,
    'num_workers' : 2,
    'seed'        : 42,
    'out_dir'     : '/content/BrainTumorAI/checkpoints/unet',
    'vis_every'   : 10,
    'device'      : 'cuda',
}
os.makedirs(CONFIG['out_dir'], exist_ok=True)
for k, v in CONFIG.items():
    print(f"  {k:15s}: {v}")

In [ ]:
import sys, time, warnings, os
import numpy as np
import torch
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
sys.path.insert(0, '/content/BrainTumorAI')

from segmentation.unet    import UNet
from segmentation.losses  import BCEDiceLoss
from segmentation.metrics import compute_all_metrics
from segmentation.dataset import BrainSegDataset

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
DEVICE = torch.device(CONFIG['device'])

def make_ds(train):
    return BrainSegDataset(
        image_dir    = CONFIG['images_dir'],
        pseudo_cache = CONFIG['pseudo_cache'],
        image_size   = CONFIG['image_size'],
        train        = train,
    )

full_ds = make_ds(train=True)
n_total = len(full_ds)
n_val   = max(1, int(n_total * CONFIG['val_split']))
n_train = n_total - n_val

gen = torch.Generator().manual_seed(CONFIG['seed'])
tr_sub, va_sub = random_split(full_ds, [n_train, n_val], generator=gen)
va_sub.dataset = make_ds(train=False)

ldr_kw = dict(batch_size=CONFIG['batch_size'],
              num_workers=CONFIG['num_workers'], pin_memory=True)
train_loader = DataLoader(tr_sub, shuffle=True,  **ldr_kw)
val_loader   = DataLoader(va_sub, shuffle=False, **ldr_kw)
print(f"Train: {n_train}  |  Val: {n_val}  |  Total: {n_total}")

model = UNet(
    in_channels  = CONFIG['in_channels'],
    base_filters = CONFIG['base_filters'],
    bilinear     = CONFIG['bilinear'],
).to(DEVICE)
print(f"U-Net parameters: {model.count_parameters():,}")

criterion = BCEDiceLoss(alpha=CONFIG['bce_alpha'])
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'], weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=8, min_lr=1e-7)

print(f"\n{'Ep':>4} | {'TrLoss':>7} {'TrDice':>7} {'TrIoU':>6} | "
      f"{'VaLoss':>7} {'VaDice':>7} {'VaIoU':>6} | {'LR':>8} | {'s':>5}")
print("-"*80)

In [ ]:
def run_epoch(loader, train):
    model.train() if train else model.eval()
    tots = {'loss':0., 'dice':0., 'iou':0.}
    n = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            imgs  = batch['image'].to(DEVICE)
            masks = batch['mask'].to(DEVICE)
            logits = model(imgs)
            loss, _ = criterion(logits, masks)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            bs  = imgs.size(0)
            met = compute_all_metrics(logits.detach(), masks, CONFIG['threshold'])
            tots['loss'] += loss.item() * bs
            tots['dice'] += met['dice']  * bs
            tots['iou']  += met['iou']   * bs
            n += bs
    return {k: v / max(n,1) for k, v in tots.items()}

history   = {'tr_loss':[],'tr_dice':[],'tr_iou':[],'val_loss':[],'val_dice':[],'val_iou':[]}
best_dice = 0.
best_ckpt = None
OUT_DIR   = CONFIG['out_dir']

for epoch in range(1, CONFIG['epochs']+1):
    t0  = time.time()
    tr  = run_epoch(train_loader, True)
    val = run_epoch(val_loader,   False)
    scheduler.step(val['dice'])
    lr_now = optimizer.param_groups[0]['lr']

    for k in ['loss','dice','iou']:
        history[f'tr_{k}'].append(tr[k])
        history[f'val_{k}'].append(val[k])

    print(f"{epoch:>4d} | {tr['loss']:>7.4f} {tr['dice']:>7.4f} {tr['iou']:>6.4f} | "
          f"{val['loss']:>7.4f} {val['dice']:>7.4f} {val['iou']:>6.4f} | "
          f"{lr_now:>8.2e} | {time.time()-t0:>5.1f}s", flush=True)

    if val['dice'] > best_dice:
        best_dice  = val['dice']
        ckpt_name  = f"unet_best_ep{epoch:03d}_dice{best_dice:.4f}.pt"
        ckpt_path  = f"{OUT_DIR}/{ckpt_name}"
        torch.save({
            'epoch'            : epoch,
            'model_state_dict' : model.state_dict(),
            'optimizer_state'  : optimizer.state_dict(),
            'val_dice'         : best_dice,
            'val_iou'          : val['iou'],
            'config'           : {
                'in_channels' : CONFIG['in_channels'],
                'base_filters': CONFIG['base_filters'],
                'bilinear'    : CONFIG['bilinear'],
                'image_size'  : CONFIG['image_size'],
            },
        }, ckpt_path)
        if best_ckpt and os.path.exists(best_ckpt) and best_ckpt != ckpt_path:
            os.remove(best_ckpt)
        best_ckpt = ckpt_path
        print(f"         *** New best: {ckpt_name}")

    if CONFIG['vis_every'] > 0 and (epoch % CONFIG['vis_every'] == 0 or epoch == 1):
        model.eval()
        fig, axes = plt.subplots(2, 4, figsize=(16,8), facecolor='#0f172a')
        mean = np.array([0.485,0.456,0.406])
        std  = np.array([0.229,0.224,0.225])
        with torch.no_grad():
            for batch in val_loader:
                imgs2 = batch['image'].to(DEVICE)
                probs = torch.sigmoid(model(imgs2)).cpu()
                for i in range(min(4, imgs2.size(0))):
                    t = imgs2[i].cpu().numpy()
                    if t.shape[0] == 1: t = np.repeat(t, 3, axis=0)
                    axes[0][i].imshow((t.transpose(1,2,0)*std+mean).clip(0,1))
                    axes[0][i].set_title(f'ep{epoch}', color='#00d4ff', fontsize=8)
                    axes[0][i].axis('off')
                    axes[1][i].imshow(probs[i,0].numpy(), cmap='RdYlGn', vmin=0, vmax=1)
                    axes[1][i].set_title(f'dice={val["dice"]:.3f}', color='#10b981', fontsize=8)
                    axes[1][i].axis('off')
                break
        plt.tight_layout()
        plt.savefig(f"{OUT_DIR}/vis_ep{epoch:03d}.png", dpi=80,
                    bbox_inches='tight', facecolor='#0f172a')
        plt.show(); plt.close()

print("="*80)
print(f"  Training complete!  Best Val Dice: {best_dice:.4f}")
print(f"  Checkpoint: {best_ckpt}")
print("="*80)

## Step 8 — Save to Drive + Download

In [ ]:
import matplotlib.pyplot as plt, os

# Re-declare in case of variable loss after restart
DRIVE_DIR = '/content/drive/MyDrive/NeuroScan_UNet'
OUT_DIR   = CONFIG['out_dir']

fig, axes = plt.subplots(1, 3, figsize=(15,4), facecolor='#0f172a')
for ax, (key, color) in zip(axes, zip(
        ['loss','dice','iou'], ['#ef4444','#10b981','#3b82f6'])):
    ax.plot(history[f'tr_{key}'],  color=color, lw=2, label='Train')
    ax.plot(history[f'val_{key}'], color=color, lw=2, ls='--', alpha=0.7, label='Val')
    ax.set_title(key.upper(), color='white', fontsize=12)
    ax.set_facecolor('#0f172a'); ax.tick_params(colors='#94a3b8')
    ax.legend(facecolor='#1e293b', labelcolor='white', fontsize=9)
    for sp in ax.spines.values(): sp.set_edgecolor('#334155')
plt.suptitle(f'U-Net Training -- Best Val Dice: {best_dice:.4f}',
             color='white', fontsize=13)
plt.tight_layout()
curves_path = f'{OUT_DIR}/training_curves.png'
plt.savefig(curves_path, dpi=120, bbox_inches='tight', facecolor='#0f172a')
plt.show()
print(f'Curves saved: {curves_path}')


In [ ]:
import shutil, glob, os

DRIVE_DIR  = '/content/drive/MyDrive/NeuroScan_UNet'
DRIVE_UNET = f'{DRIVE_DIR}/checkpoints_unet'
OUT_DIR    = CONFIG['out_dir']
os.makedirs(DRIVE_UNET, exist_ok=True)

if best_ckpt and os.path.exists(best_ckpt):
    shutil.copy2(best_ckpt, DRIVE_UNET)
    print(f'Checkpoint -> Drive: {os.path.basename(best_ckpt)}')
else:
    print('WARNING: best_ckpt not found!')

if os.path.exists(curves_path):
    shutil.copy2(curves_path, DRIVE_UNET)
    print('Training curves -> Drive')

vis_files = glob.glob(f'{OUT_DIR}/vis_ep*.png')
for vf in vis_files:
    shutil.copy2(vf, DRIVE_UNET)
print(f'{len(vis_files)} visualisation images -> Drive')

print(f'\nAll saved to: {DRIVE_UNET}')
print('(Persists after Colab disconnects)')
print('\nFiles in Drive:')
for f in sorted(glob.glob(f'{DRIVE_UNET}/*')):
    print(f'  {os.path.basename(f):50s} {os.path.getsize(f)//1024:5d} KB')


In [ ]:
from google.colab import files
import os

if best_ckpt and os.path.exists(best_ckpt):
    print(f'Downloading: {os.path.basename(best_ckpt)}')
    print('Download dialog will appear in your browser...')
    files.download(best_ckpt)
    print('\nDone! Place the .pt file in:')
    print('  checkpoints/unet/   on your local PC')
else:
    # Fallback: download from Drive
    DRIVE_UNET = '/content/drive/MyDrive/NeuroScan_UNet/checkpoints_unet'
    pts = sorted(__import__('glob').glob(f'{DRIVE_UNET}/unet*.pt'))
    if pts:
        print(f'Downloading from Drive: {os.path.basename(pts[-1])}')
        files.download(pts[-1])
    else:
        print('ERROR: No checkpoint found. Check Drive or re-run training.')


## Step 9 — Deploy Locally

```
1. Move downloaded .pt to:
   C:\Users\HP\Desktop\PRO(B)\BrainTumorAI\checkpoints\unet\

2. Run:
   cd "C:\Users\HP\Desktop\PRO(B)\BrainTumorAI"
   python app.py

3. Open http://localhost:7862
   Upload MRI → expand "Tumour Segmentation (U-Net)"
   Badge shows: 🟢 Live U-Net
```

In [ ]:
import glob, os, torch
print("="*60)
print("  TRAINING COMPLETE")
print("="*60)
print(f"  Best Val Dice : {best_dice:.4f}")
print(f"  Checkpoint    : {os.path.basename(best_ckpt)}")
print(f"  Epochs        : {CONFIG['epochs']}")
print(f"  Image size    : {CONFIG['image_size']}x{CONFIG['image_size']}")
print(f"  Model params  : {model.count_parameters():,}")
print(f"  GPU           : {torch.cuda.get_device_name(0)}")
print()
for f in sorted(glob.glob(f'{DRIVE_UNET}/*')):
    print(f"  {os.path.basename(f):45s} {os.path.getsize(f)//1024:5d} KB")
print("="*60)